In [2]:
import numpy as np
import pandas as pd
import os


# ============================================================
# SETTINGS
# ============================================================

regions = range(1, 8)

methods = [
    'USGS',
    'Furey',
    'Eckhardt',
    'LocalHYSEP'
]

models = [
    'HBV',
    'HBV_capillary',
    'marrmot_PRMS'
]

test = 'PUR'

objectives = ['Single', 'Multi']

# Output directory
output_dir = "/home/s0shar47/scratch/RandomForestData"

os.makedirs(output_dir, exist_ok=True)


# ============================================================
# ATTRIBUTE NAMES
# ============================================================

attribute_names = [
    "p_mean",
    "pet_mean",
    "p_seasonality",
    "frac_snow",
    "aridity",
    "high_prec_freq",
    "high_prec_dur",
    "low_prec_freq",
    "low_prec_dur",
    "elev_mean",
    "slope_mean",
    "area_gages2",
    "frac_forest",
    "lai_max",
    "lai_diff",
    "gvf_max",
    "gvf_diff",
    "dom_land_cover_frac",
    "dom_land_cover",
    "root_depth_50",
    "soil_depth_pelletier",
    "soil_depth_statsgo",
    "soil_porosity",
    "soil_conductivity",
    "max_water_content",
    "sand_frac",
    "silt_frac",
    "clay_frac",
    "geol_1st_class",
    "glim_1st_class_frac",
    "geol_2nd_class",
    "glim_2nd_class_frac",
    "carbonate_rocks_frac",
    "geol_porostiy",
    "geol_permeability"
]


# ============================================================
# MODEL DYNAMICS
# ============================================================

dyn_dict = {
    "HBV": "parBETA_parBETAET_",
    "HBV_capillary": "parBETA_parBETAET_parK0_parUZL_",
    "marrmot_PRMS": "alpha_cgw_scx_tt_"
}


# ============================================================
# NSE FUNCTION
# ============================================================

def calc_nse(obs, sim):

    mask = np.isfinite(obs) & np.isfinite(sim)

    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    den = np.sum((obs - np.mean(obs)) ** 2)

    if den == 0:
        return np.nan

    return 1 - np.sum((obs - sim) ** 2) / den


# ============================================================
# STEP 1
# COLLECT GAGE IDs + ATTRIBUTES FROM ALL 7 REGIONS
# ============================================================

all_attributes = []

print("\n" + "=" * 80)
print("COLLECTING BASIN ATTRIBUTES")
print("=" * 80)

for region in regions:

    print(f"\nProcessing Region_{region}")

    forcing_path = (
        f"/scratch/data/PUR_DATA/test/Region_{region}/"
        f"daymet_19800101_20141231_forcing_671_baseflow_mmday.npy"
    )

    attr_path = (
        f"/scratch/data/PUR_DATA/test/Region_{region}/"
        f"daymet_19800101_20141231_attr_671.npy"
    )

    # --------------------------------------------------------
    # Load
    # --------------------------------------------------------

    forcing = np.load(forcing_path)
    attributes = np.load(attr_path)

    # --------------------------------------------------------
    # GAGE IDs
    # forcing[:, 0, 14]
    # --------------------------------------------------------

    gage_ids = forcing[:, 0, 14].astype(int)

    print(f"  Number of basins: {len(gage_ids)}")
    print(f"  Attribute shape: {attributes.shape}")

    # --------------------------------------------------------
    # Check dimensions
    # --------------------------------------------------------

    if len(gage_ids) != attributes.shape[0]:

        raise ValueError(
            f"Region_{region}: GAGE ID count "
            f"({len(gage_ids)}) does not match attribute count "
            f"({attributes.shape[0]})"
        )

    # --------------------------------------------------------
    # Create dataframe
    # --------------------------------------------------------

    region_df = pd.DataFrame(
        attributes,
        columns=attribute_names
    )

    region_df.insert(
        0,
        "GAGE_ID",
        gage_ids
    )

    region_df["Region"] = region

    all_attributes.append(region_df)


# ============================================================
# COMBINE ALL REGIONS
# ============================================================

attributes_all = pd.concat(
    all_attributes,
    ignore_index=True
)

print("\nTotal basin records across regions:")
print(len(attributes_all))

print(
    "Unique GAGE IDs:",
    attributes_all["GAGE_ID"].nunique()
)


# ============================================================
# REMOVE DUPLICATE GAGE IDS
# ============================================================

attributes_all = attributes_all.drop_duplicates(
    subset="GAGE_ID",
    keep="first"
).reset_index(drop=True)


print(
    "Unique basins after removing duplicates:",
    len(attributes_all)
)


# ============================================================
# STEP 2
# DETERMINE THE 531 BASINS FROM THE MODEL RESULTS
# ============================================================

# We use the observed flow files to identify exactly which
# basins are included in the 531-basin model dataset.

model_gage_ids = []


print("\n" + "=" * 80)
print("IDENTIFYING MODEL BASINS")
print("=" * 80)


for region in regions:

    # --------------------------------------------------------
    # The observed forcing contains the actual GAGE IDs.
    # --------------------------------------------------------

    forcing_path = (
        f"/scratch/data/PUR_DATA/test/Region_{region}/"
        f"daymet_19800101_20141231_forcing_671_baseflow_mmday.npy"
    )

    forcing = np.load(forcing_path)

    gage_ids = forcing[:, 0, 14].astype(int)

    # --------------------------------------------------------
    # The actual 531-basin model selection is assumed to be
    # represented by the basins present in the model output.
    #
    # We identify these from the observed flow output.
    # --------------------------------------------------------

    folder = (
        f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/"
        f"LocalHYSEP_Multi/Region_{region}/"
    )

    model_name = (
        "LSTM_HBV_E50_R365_B100_H256_"
        "tr1989_1999_n16_111111"
    )

    dyn = dyn_dict["HBV"]

    obs_path = os.path.join(
        folder,
        model_name,
        dyn,
        "ts1989_1999",
        "00060_Mean.npy"
    )

    if not os.path.exists(obs_path):

        print(
            f"WARNING: Could not find {obs_path}"
        )

        continue

    obs_flow = np.load(obs_path)
    obs_flow = np.squeeze(obs_flow)

    n_basins = obs_flow.shape[1]

    print(
        f"Region_{region}: {n_basins} model basins"
    )

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # The model basin ordering must correspond to the forcing
    # basin ordering.
    # --------------------------------------------------------

    region_gage_ids = gage_ids[:n_basins]

    model_gage_ids.extend(
        region_gage_ids.tolist()
    )


# ============================================================
# UNIQUE 531 BASINS
# ============================================================

model_gage_ids = np.array(
    list(dict.fromkeys(model_gage_ids)),
    dtype=int
)

print("\nNumber of model basins:")
print(len(model_gage_ids))


# ============================================================
# SELECT ATTRIBUTES FOR THE MODEL BASINS
# ============================================================

attributes_531 = attributes_all[
    attributes_all["GAGE_ID"].isin(model_gage_ids)
].copy()


# Preserve the model basin order
attributes_531["order"] = (
    attributes_531["GAGE_ID"]
    .map(
        {
            gage: i
            for i, gage in enumerate(model_gage_ids)
        }
    )
)

attributes_531 = (
    attributes_531
    .sort_values("order")
    .drop(columns="order")
    .reset_index(drop=True)
)


# ============================================================
# CHECK
# ============================================================

print("\nFinal attribute dataset:")
print(
    "Number of basins:",
    len(attributes_531)
)

print(
    "Number of attributes:",
    len(attribute_names)
)


# ============================================================
# SAVE ATTRIBUTE CSV
# ============================================================

attribute_output = os.path.join(
    output_dir,
    "basin_attributes_531.csv"
)

attributes_531.to_csv(
    attribute_output,
    index=False
)

print(
    f"\nSaved attributes:\n{attribute_output}"
)


# ============================================================
# STEP 3
# CALCULATE AND SAVE NSE FOR EVERY METHOD + MODEL
# ============================================================

print("\n" + "=" * 80)
print("CALCULATING BASIN-WISE NSE")
print("=" * 80)


for method in methods:

    for model in models:

        print("\n" + "-" * 80)

        print(
            f"Method = {method} | Model = {model}"
        )

        dyn = dyn_dict[model]

        # ----------------------------------------------------
        # Containers
        # ----------------------------------------------------

        results = []

        # ====================================================
        # LOOP OVER REGIONS
        # ====================================================

        for region in regions:

            # ------------------------------------------------
            # Paths
            # ------------------------------------------------

            folder = (
                f"/home/s0shar47/scratch/"
                f"OUTPUTS_Paper1/PUR/"
                f"{method}_Multi/Region_{region}/"
            )

            sfolder = (
                f"/home/s0shar47/scratch/"
                f"OUTPUTS_Paper1/PUR/"
                f"OUTPUTS_PUR_Single/Region_{region}/"
            )

            model_name = (
                f"LSTM_{model}_E50_R365_B100_H256_"
                f"tr1989_1999_n16_111111"
            )

            # ------------------------------------------------
            # Observed flow
            # ------------------------------------------------

            obs_path = os.path.join(
                folder,
                model_name,
                dyn,
                "ts1989_1999",
                "00060_Mean.npy"
            )

            # ------------------------------------------------
            # Old simulation
            # ------------------------------------------------

            old_path = os.path.join(
                sfolder,
                model_name,
                dyn,
                "ts1989_1999",
                "flow_sim.npy"
            )

            # ------------------------------------------------
            # New simulation
            # ------------------------------------------------

            new_path = os.path.join(
                folder,
                model_name,
                dyn,
                "ts1989_1999",
                "flow_sim.npy"
            )

            # ------------------------------------------------
            # Check files
            # ------------------------------------------------

            if not os.path.exists(obs_path):

                print(
                    f"  Missing OBS: Region_{region}"
                )

                continue

            if not os.path.exists(old_path):

                print(
                    f"  Missing OLD: Region_{region}"
                )

                continue

            if not os.path.exists(new_path):

                print(
                    f"  Missing NEW: Region_{region}"
                )

                continue

            # ------------------------------------------------
            # Load
            # ------------------------------------------------

            obs_flow = np.squeeze(
                np.load(obs_path)
            )

            sim_old = np.squeeze(
                np.load(old_path)
            )

            sim_new = np.squeeze(
                np.load(new_path)
            )

            # ------------------------------------------------
            # Load GAGE IDs
            # ------------------------------------------------

            forcing_path = (
                f"/scratch/data/PUR_DATA/test/"
                f"Region_{region}/"
                f"daymet_19800101_20141231_"
                f"forcing_671_baseflow_mmday.npy"
            )

            forcing = np.load(forcing_path)

            gage_ids = forcing[
                :, 0, 14
            ].astype(int)

            # ------------------------------------------------
            # Sanity check
            # ------------------------------------------------

            if obs_flow.shape[1] != len(gage_ids):

                print(
                    f"WARNING Region_{region}: "
                    f"flow basins = {obs_flow.shape[1]}, "
                    f"forcing basins = {len(gage_ids)}"
                )

                # If model output is a subset, use the
                # corresponding first N basin IDs.
                gage_ids = gage_ids[
                    :obs_flow.shape[1]
                ]

            # =================================================
            # LOOP OVER BASINS
            # =================================================

            for k in range(obs_flow.shape[1]):

                obs = obs_flow[:, k]

                old = sim_old[:, k]

                new = sim_new[:, k]

                # ------------------------------------------------
                # NSE
                # ------------------------------------------------

                nse_old = calc_nse(
                    obs,
                    old
                )

                nse_new = calc_nse(
                    obs,
                    new
                )

                # ------------------------------------------------
                # Store
                # ------------------------------------------------

                results.append({

                    "GAGE_ID": gage_ids[k],

                    "Region": region,

                    "NSE_old": nse_old,

                    "NSE_new": nse_new,

                    "NSE_improvement":
                        nse_new - nse_old
                })

        # ====================================================
        # DATAFRAME
        # ====================================================

        df_nse = pd.DataFrame(results)

        # ----------------------------------------------------
        # Remove duplicate GAGE IDs if necessary
        # ----------------------------------------------------

        df_nse = (
            df_nse
            .drop_duplicates(
                subset="GAGE_ID",
                keep="first"
            )
            .reset_index(drop=True)
        )

        # ----------------------------------------------------
        # Sort
        # ----------------------------------------------------

        df_nse = df_nse.sort_values(
            "GAGE_ID"
        ).reset_index(drop=True)

        # ====================================================
        # SAVE
        # ====================================================

        filename = (
            f"NSE_{method}_{model}.csv"
        )

        output_path = os.path.join(
            output_dir,
            filename
        )

        df_nse.to_csv(
            output_path,
            index=False
        )

        # ====================================================
        # SUMMARY
        # ====================================================

        print(
            f"Saved: {output_path}"
        )

        print(
            f"Basins: {len(df_nse)}"
        )

        print(
            f"Median NSE old: "
            f"{np.nanmedian(df_nse['NSE_old']):.4f}"
        )

        print(
            f"Median NSE new: "
            f"{np.nanmedian(df_nse['NSE_new']):.4f}"
        )

        print(
            f"Median improvement: "
            f"{np.nanmedian(df_nse['NSE_improvement']):.4f}"
        )


print("\n" + "=" * 80)
print("DONE")
print("=" * 80)


COLLECTING BASIN ATTRIBUTES

Processing Region_1
  Number of basins: 92
  Attribute shape: (92, 35)

Processing Region_2
  Number of basins: 95
  Attribute shape: (95, 35)

Processing Region_3
  Number of basins: 93
  Attribute shape: (93, 35)

Processing Region_4
  Number of basins: 51
  Attribute shape: (51, 35)

Processing Region_5
  Number of basins: 68
  Attribute shape: (68, 35)

Processing Region_6
  Number of basins: 60
  Attribute shape: (60, 35)

Processing Region_7
  Number of basins: 72
  Attribute shape: (72, 35)

Total basin records across regions:
531
Unique GAGE IDs: 531
Unique basins after removing duplicates: 531

IDENTIFYING MODEL BASINS
Region_1: 92 model basins
Region_2: 95 model basins
Region_3: 93 model basins
Region_4: 51 model basins
Region_5: 68 model basins
Region_6: 60 model basins
Region_7: 72 model basins

Number of model basins:
531

Final attribute dataset:
Number of basins: 531
Number of attributes: 35

Saved attributes:
/home/s0shar47/scratch/RandomFo

In [5]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.inspection import permutation_importance


# ============================================================
# SETTINGS
# ============================================================

DATA_DIR = "/home/s0shar47/scratch/RandomForestData"

ATTRIBUTE_FILE = os.path.join(
    DATA_DIR,
    "basin_attributes_531.csv"
)

# ------------------------------------------------------------
# Analyze all three targets separately
# ------------------------------------------------------------

targets = [
    "NSE_old",
    "NSE_new",
    "NSE_improvement"
]


# ============================================================
# ATTRIBUTE NAMES
# ============================================================

attribute_names = [
    "p_mean",
    "pet_mean",
    "p_seasonality",
    "frac_snow",
    "aridity",
    "high_prec_freq",
    "high_prec_dur",
    "low_prec_freq",
    "low_prec_dur",
    "elev_mean",
    "slope_mean",
    "area_gages2",
    "frac_forest",
    "lai_max",
    "lai_diff",
    "gvf_max",
    "gvf_diff",
    "dom_land_cover_frac",
    "dom_land_cover",
    "root_depth_50",
    "soil_depth_pelletier",
    "soil_depth_statsgo",
    "soil_porosity",
    "soil_conductivity",
    "max_water_content",
    "sand_frac",
    "silt_frac",
    "clay_frac",
    "geol_1st_class",
    "glim_1st_class_frac",
    "geol_2nd_class",
    "glim_2nd_class_frac",
    "carbonate_rocks_frac",
    "geol_porostiy",
    "geol_permeability"
]


# ============================================================
# LOAD ATTRIBUTES
# ============================================================

attributes = pd.read_csv(
    ATTRIBUTE_FILE
)

print("Attribute dataset:", attributes.shape)

print(
    "Unique basins:",
    attributes["GAGE_ID"].nunique()
)


# ============================================================
# FIND NSE FILES
# ============================================================

nse_files = [
    f for f in os.listdir(DATA_DIR)
    if f.startswith("NSE_")
    and f.endswith(".csv")
]

print("\nNSE files found:")

for f in nse_files:
    print(" ", f)


# ============================================================
# BASE RESULTS DIRECTORY
# ============================================================

RESULTS_DIR = os.path.join(
    DATA_DIR,
    "RandomForestResults"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)


# ============================================================
# RANDOM FOREST FUNCTION
# ============================================================

def run_random_forest(
    nse_file,
    target
):

    print("\n" + "=" * 80)
    print(
        f"File   : {nse_file}"
    )
    print(
        f"Target : {target}"
    )
    print("=" * 80)


    # ========================================================
    # LOAD NSE DATA
    # ========================================================

    nse = pd.read_csv(
        os.path.join(
            DATA_DIR,
            nse_file
        )
    )


    # ========================================================
    # MERGE WITH ATTRIBUTES
    # ========================================================

    df = pd.merge(
        nse,
        attributes[
            ["GAGE_ID"] + attribute_names
        ],
        on="GAGE_ID",
        how="inner"
    )

    print(
        "Merged dataset:",
        df.shape
    )


    # ========================================================
    # SELECT VARIABLES
    # ========================================================

    df = df[
        ["GAGE_ID", target] + attribute_names
    ].copy()


    # ========================================================
    # CLEAN DATA
    # ========================================================

    df = df.replace(
        [np.inf, -np.inf],
        np.nan
    )


    # Remove basins without target
    df = df.dropna(
        subset=[target]
    )


    # Fill missing attributes using median
    for col in attribute_names:

        df[col] = df[col].fillna(
            df[col].median()
        )


    print(
        "Basins used:",
        len(df)
    )


    # ========================================================
    # X AND Y
    # ========================================================

    X = df[attribute_names]

    y = df[target]


    print(
        f"{target} mean = {y.mean():.4f}"
    )

    print(
        f"{target} std  = {y.std():.4f}"
    )


    # ========================================================
    # RANDOM FOREST
    # ========================================================

    rf = RandomForestRegressor(

        n_estimators=1000,

        max_features="sqrt",

        min_samples_leaf=5,

        random_state=42,

        n_jobs=-1
    )


    # ========================================================
    # TRAIN MODEL
    # ========================================================

    rf.fit(
        X,
        y
    )


    # ========================================================
    # 5-FOLD CROSS VALIDATION
    # ========================================================

    cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )


    cv_r2 = cross_val_score(
        rf,
        X,
        y,
        cv=cv,
        scoring="r2",
        n_jobs=-1
    )


    cv_rmse = np.sqrt(
        -cross_val_score(
            rf,
            X,
            y,
            cv=cv,
            scoring="neg_mean_squared_error",
            n_jobs=-1
        )
    )


    print("\nCross-validation:")

    print(
        f"R2   = {cv_r2.mean():.4f} "
        f"+/- {cv_r2.std():.4f}"
    )

    print(
        f"RMSE = {cv_rmse.mean():.4f} "
        f"+/- {cv_rmse.std():.4f}"
    )


    # ========================================================
    # RANDOM FOREST FEATURE IMPORTANCE
    # ========================================================

    rf_importance = pd.DataFrame({

        "Attribute":
            attribute_names,

        "RF_importance":
            rf.feature_importances_

    })


    # ========================================================
    # PERMUTATION IMPORTANCE
    # ========================================================

    print(
        "\nCalculating permutation importance..."
    )


    perm = permutation_importance(

        rf,

        X,

        y,

        n_repeats=30,

        random_state=42,

        n_jobs=-1,

        scoring="r2"
    )


    permutation_df = pd.DataFrame({

        "Attribute":
            attribute_names,

        "Permutation_importance":
            perm.importances_mean,

        "Permutation_std":
            perm.importances_std
    })


    # ========================================================
    # COMBINE IMPORTANCE
    # ========================================================

    importance = pd.merge(

        rf_importance,

        permutation_df,

        on="Attribute"
    )


    importance = importance.sort_values(

        "Permutation_importance",

        ascending=False
    ).reset_index(drop=True)


    # ========================================================
    # GET METHOD + MODEL NAME
    # ========================================================

    base_name = (

        nse_file
        .replace("NSE_", "")
        .replace(".csv", "")
    )


    # ========================================================
    # TARGET-SPECIFIC FOLDER
    # ========================================================

    target_dir = os.path.join(

        RESULTS_DIR,

        target
    )

    os.makedirs(

        target_dir,

        exist_ok=True
    )


    # ========================================================
    # SAVE IMPORTANCE CSV
    # ========================================================

    importance_file = os.path.join(

        target_dir,

        f"importance_{base_name}.csv"
    )


    importance.to_csv(

        importance_file,

        index=False
    )


    # ========================================================
    # SAVE MODEL PERFORMANCE
    # ========================================================

    performance = pd.DataFrame({

        "Target": [target],

        "Method_Model": [base_name],

        "N_basins": [len(df)],

        "CV_R2_mean": [cv_r2.mean()],

        "CV_R2_std": [cv_r2.std()],

        "CV_RMSE_mean": [cv_rmse.mean()],

        "CV_RMSE_std": [cv_rmse.std()]
    })


    performance_file = os.path.join(

        target_dir,

        f"performance_{base_name}.csv"
    )


    performance.to_csv(

        performance_file,

        index=False
    )


    # ========================================================
    # PRINT TOP 15
    # ========================================================

    print("\nTop 15 attributes:")

    print(

        importance.head(15).to_string(
            index=False
        )

    )


    # ========================================================
    # PLOT TOP 15
    # ========================================================

    top = (

        importance
        .head(15)
        .sort_values(
            "Permutation_importance"
        )
    )


    plt.figure(
        figsize=(9, 7)
    )


    plt.barh(

        top["Attribute"],

        top["Permutation_importance"]
    )


    plt.xlabel(
        "Permutation importance (Δ R²)"
    )

    plt.ylabel(
        "Basin attribute"
    )

    plt.title(

        f"{target}: {base_name}"
    )


    plt.tight_layout()


    plot_file = os.path.join(

        target_dir,

        f"importance_{base_name}.png"
    )


    plt.savefig(

        plot_file,

        dpi=300,

        bbox_inches="tight"
    )


    plt.close()


    return importance, performance


# ============================================================
# RUN ALL TARGETS × ALL NSE FILES
# ============================================================

all_importance = []

all_performance = []


for target in targets:

    print("\n\n")

    print("#" * 90)

    print(
        f"TARGET = {target}"
    )

    print("#" * 90)


    for nse_file in nse_files:

        try:

            importance, performance = (

                run_random_forest(

                    nse_file,

                    target

                )

            )


            # Add target
            importance["Target"] = target

            all_importance.append(
                importance
            )

            all_performance.append(
                performance
            )


        except Exception as e:

            print(
                f"\nERROR:"
                f" {nse_file} | "
                f"{target}"
            )

            print(e)


# ============================================================
# SAVE COMBINED FILE FOR EACH TARGET
# ============================================================

for target in targets:

    # --------------------------------------------------------
    # Importance
    # --------------------------------------------------------

    target_results = [

        x for x in all_importance

        if x["Target"].iloc[0] == target
    ]


    if len(target_results) > 0:

        combined = pd.concat(

            target_results,

            ignore_index=True
        )


        target_dir = os.path.join(

            RESULTS_DIR,

            target
        )


        combined.to_csv(

            os.path.join(

                target_dir,

                f"ALL_{target}_importance.csv"
            ),

            index=False
        )


    # --------------------------------------------------------
    # Performance
    # --------------------------------------------------------

    target_perf = [

        x for x in all_performance

        if x["Target"].iloc[0] == target
    ]


    if len(target_perf) > 0:

        performance_combined = pd.concat(

            target_perf,

            ignore_index=True
        )


        target_dir = os.path.join(

            RESULTS_DIR,

            target
        )


        performance_combined.to_csv(

            os.path.join(

                target_dir,

                f"ALL_{target}_performance.csv"
            ),

            index=False
        )


print("\n" + "=" * 90)

print("RANDOM FOREST ANALYSIS COMPLETE")

print("=" * 90)

print(
    f"Results saved in:\n{RESULTS_DIR}"
)

Attribute dataset: (531, 37)
Unique basins: 531

NSE files found:
  NSE_LocalHYSEP_marrmot_PRMS.csv
  NSE_Eckhardt_HBV_capillary.csv
  NSE_Eckhardt_marrmot_PRMS.csv
  NSE_USGS_HBV.csv
  NSE_Eckhardt_HBV.csv
  NSE_Furey_marrmot_PRMS.csv
  NSE_LocalHYSEP_HBV.csv
  NSE_LocalHYSEP_HBV_capillary.csv
  NSE_USGS_HBV_capillary.csv
  NSE_Furey_HBV.csv
  NSE_USGS_marrmot_PRMS.csv
  NSE_Furey_HBV_capillary.csv



##########################################################################################
TARGET = NSE_old
##########################################################################################

File   : NSE_LocalHYSEP_marrmot_PRMS.csv
Target : NSE_old
Merged dataset: (531, 40)
Basins used: 531
NSE_old mean = 0.3518
NSE_old std  = 1.5499

Cross-validation:
R2   = 0.0958 +/- 0.0557
RMSE = 1.2688 +/- 0.7970

Calculating permutation importance...

Top 15 attributes:
           Attribute  RF_importance  Permutation_importance  Permutation_std
         frac_forest       0.060133         